In [12]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import scipy.io as sio

def reward_categorized_psth(spikes_path, laser_times_path, save_folder, laser_type, csv_path=None, qc_data=None, 
                        bin_size=0.005, time_window=(-1, 1.5), type_file='pdf', 
                        laser_delay=0.5, laser_duration=0.5,
                        min_trials=3, min_spikes=5, 
                        process_individual_conditions=True):
    """
    generate PSTH plots with categories based on reward outcomes.
    """
    os.makedirs(save_folder, exist_ok=True)
    spikes = np.load(spikes_path, allow_pickle=True)
    mat_data = sio.loadmat(laser_times_path)
    trial_data_df = None
    
    trial_data_df = csv_path
    
    # load from csv trial data
    if trial_data_df is not None:
        print("Categorizing trials using CSV data...")
        
        reward_right_laser_times = []
        reward_left_laser_times = []
        nonreward_right_laser_times = []
        nonreward_left_laser_times = []
        
        reward_right_control_times = []
        reward_left_control_times = []
        nonreward_right_control_times = []
        nonreward_left_control_times = []
        
        # CHANGE HERE FOR SPLIT TRIAL, nvm
        for _, trial in trial_data_df.iterrows(): 

            try:
                trial_time = trial['TimeStart']
                is_laser = trial['IsLaserTrial'] == 1
                is_right = trial['TrialSide'] == 'Right'
                is_rewarded = trial['RMI'] == 'reward'
                
                if is_laser:
                    if is_right:
                        if is_rewarded:
                            reward_right_laser_times.append(trial_time)
                        else:
                            nonreward_right_laser_times.append(trial_time)
                    else:  # Left trial
                        if is_rewarded:
                            reward_left_laser_times.append(trial_time)
                        else:
                            nonreward_left_laser_times.append(trial_time)
                else:  # Control trial
                    if is_right:
                        if is_rewarded:
                            reward_right_control_times.append(trial_time)
                        else:
                            nonreward_right_control_times.append(trial_time)
                    else:  # Left trial
                        if is_rewarded:
                            reward_left_control_times.append(trial_time)
                        else:
                            nonreward_left_control_times.append(trial_time)
            except Exception as e:
                print(f"Error processing trial: {e}")
                # Continue with the next trial
                continue
        
        reward_right_laser_times = np.array(reward_right_laser_times)
        reward_left_laser_times = np.array(reward_left_laser_times)
        nonreward_right_laser_times = np.array(nonreward_right_laser_times)
        nonreward_left_laser_times = np.array(nonreward_left_laser_times)
        
        reward_right_control_times = np.array(reward_right_control_times)
        reward_left_control_times = np.array(reward_left_control_times)
        nonreward_right_control_times = np.array(nonreward_right_control_times)
        nonreward_left_control_times = np.array(nonreward_left_control_times)
        
        all_right_laser = np.concatenate([reward_right_laser_times, nonreward_right_laser_times])
        all_left_laser = np.concatenate([reward_left_laser_times, nonreward_left_laser_times])
        all_right_control = np.concatenate([reward_right_control_times, nonreward_right_control_times])
        all_left_control = np.concatenate([reward_left_control_times, nonreward_left_control_times])
        
        reward_right_laser = reward_right_laser_times
        reward_left_laser = reward_left_laser_times
        nonreward_right_laser = nonreward_right_laser_times
        nonreward_left_laser = nonreward_left_laser_times
        
        reward_right_control = reward_right_control_times
        reward_left_control = reward_left_control_times
        nonreward_right_control = nonreward_right_control_times
        nonreward_left_control = nonreward_left_control_times
        
        # print trial counts for each category
        print(f"Trial categorization summary:")
        print(f"  Right laser trials: {len(all_right_laser)} total")
        print(f"    Reward: {len(reward_right_laser_times)}")
        print(f"    Non-reward: {len(nonreward_right_laser_times)}")
        print(f"  Left laser trials: {len(all_left_laser)} total")
        print(f"    Reward: {len(reward_left_laser_times)}")
        print(f"    Non-reward: {len(nonreward_left_laser_times)}")
        print(f"  Right control trials: {len(all_right_control)} total")
        print(f"    Reward: {len(reward_right_control_times)}")
        print(f"    Non-reward: {len(nonreward_right_control_times)}")
        print(f"  Left control trials: {len(all_left_control)} total")
        print(f"    Reward: {len(reward_left_control_times)}")
        print(f"    Non-reward: {len(nonreward_left_control_times)}")
    else:
        print("No trial data available - using event-based matching")
        
        if len(right_sounds) == 0 or len(left_sounds) == 0:
            raise ValueError("No sound data available in MAT file and no CSV data provided")

        laser_times_right = []
        sound_times_right_laser = []
        for right_sound in right_sounds:
            closest_idx = np.argmin(np.abs(laser_times - right_sound))
            closest_time = laser_times[closest_idx]
            if np.abs(closest_time - right_sound) < 1:  # 1s tolerance
                laser_times_right.append(closest_time)
                sound_times_right_laser.append(right_sound)
        
        laser_times_right = np.array(laser_times_right)
        sound_times_right_laser = np.array(sound_times_right_laser)

        # left sound trials with laser
        laser_times_left = []
        sound_times_left_laser = []
        for left_sound in left_sounds:
            closest_idx = np.argmin(np.abs(laser_times - left_sound))
            closest_time = laser_times[closest_idx]
            if np.abs(closest_time - left_sound) < 1:  # 1s tolerance
                laser_times_left.append(closest_time)
                sound_times_left_laser.append(left_sound)
        
        laser_times_left = np.array(laser_times_left)
        sound_times_left_laser = np.array(sound_times_left_laser)

        # control trials (sounds without laser)
        control_times_left = []
        for left_sound in left_sounds:
            if left_sound not in sound_times_left_laser:
                control_times_left.append(left_sound)
        control_times_left = np.array(control_times_left)

        control_times_right = []
        for right_sound in right_sounds:
            if right_sound not in sound_times_right_laser:
                control_times_right.append(right_sound)
        control_times_right = np.array(control_times_right)
        
        # use these for the 'all' category
        all_right_laser = sound_times_right_laser
        all_left_laser = sound_times_left_laser
        all_right_control = control_times_right
        all_left_control = control_times_left
        
        # empty arrays for reward categories - these are the variables used in process_category function
        reward_right_laser = np.array([])
        reward_left_laser = np.array([])
        nonreward_right_laser = np.array([])
        nonreward_left_laser = np.array([])
        
        reward_right_control = np.array([])
        reward_left_control = np.array([])
        nonreward_right_control = np.array([])
        nonreward_left_control = np.array([])
        
        # print trial counts
        print(f"Trial counts:")
        print(f"  Right with laser: {len(all_right_laser)}")
        print(f"  Left with laser: {len(all_left_laser)}")
        print(f"  Right control: {len(all_right_control)}")
        print(f"  Left control: {len(all_left_control)}")
        print("  (No reward/nonreward categorization available)")

    # Load laser times and trial times from MAT file (needed even with CSV)
    try:
        laser_times = mat_data[laser_type][0, 0]['Ts'].flatten()
        print(f"Successfully loaded {len(laser_times)} laser timestamps")
        
        # try to load right and left sounds
        if 'right_sounds_evt07' in mat_data and 'left_sounds_evt08' in mat_data:
            try:
                right_sounds = mat_data['right_sounds_evt07'][0, 0]['Ts'].flatten()
                left_sounds = mat_data['left_sounds_evt08'][0, 0]['Ts'].flatten()
                print(f"Successfully loaded {len(right_sounds)} right sounds and {len(left_sounds)} left sounds")
            except (KeyError, IndexError, AttributeError) as e:
                print(f"Error accessing sound data fields: {e}")
                right_sounds = np.array([])
                left_sounds = np.array([])
        else:
            print("Warning: 'right_sounds_evt07' or 'left_sounds_evt08' not found in MAT file")
            right_sounds = np.array([])
            left_sounds = np.array([])
    except (KeyError, IndexError, AttributeError) as e:
        print(f"Error loading laser times: {e}")
        print("Available fields in MAT file:")
        for key in mat_data.keys():
            if not key.startswith('__'):
                print(f"  - {key}")
        raise ValueError(f"Could not load required data from MAT file: {e}")
        
    if len(laser_times) == 0:
        raise ValueError("No laser timestamps found in MAT file")
    
    if len(right_sounds) == 0 and len(left_sounds) == 0 and trial_data_df is not None:
        print("No sound data found in MAT file, will rely solely on CSV data for categorization")

    # unit IDs to analyze
    if qc_data is not None:
        qc = pd.read_csv(qc_data)
        units = qc.iloc[:, 0].values
    else:
        units = np.unique(spikes['unit_index'])

    # align spikes to sound times
    def align_spikes_to_sound(sound_times, neuron_spikes, time_window):
        if len(sound_times) == 0:
            return [], 0
            
        aligned_spikes = []
        spikes_count = 0
        for st in sound_times:
            spike_window = neuron_spikes[(neuron_spikes >= st + time_window[0]) & 
                                         (neuron_spikes <= st + time_window[1])]
            
            spikes_count += len(spike_window)
            spike_window_aligned = spike_window - st
            aligned_spikes.append(spike_window_aligned)
        return aligned_spikes, spikes_count

    # PSTH function 
    def compute_psth(aligned_spikes, bin_size, time_window):
        bin_edges = np.arange(time_window[0], time_window[1] + bin_size, bin_size)
        num_bins = len(bin_edges) - 1
        num_trials = len(aligned_spikes)
        
        if num_trials == 0:
            # return empty arrays if no trials
            return bin_edges, np.zeros(num_bins), np.zeros(num_bins), 0
            
        spike_counts = np.zeros((num_trials, num_bins))
        
        for i, trial_spikes in enumerate(aligned_spikes):
            counts, _ = np.histogram(trial_spikes, bins=bin_edges)
            spike_counts[i, :] = counts
            
        # convert counts to firing rates (spikes per second)
        firing_rates = spike_counts / bin_size
        
        # compute mean and standard error across trials
        mean_rates = np.mean(firing_rates, axis=0)
        std_rates = np.std(firing_rates, axis=0) / np.sqrt(num_trials) if num_trials > 1 else np.zeros(num_bins)
        
        return bin_edges, mean_rates, std_rates, num_trials

    # calcuate area under curve during laser period
    def calculate_laser_area(bin_centers, mean_rates, laser_onset=0.5, laser_duration=0.5):
        laser_start_idx = np.searchsorted(bin_centers, laser_onset)
        laser_end_idx = np.searchsorted(bin_centers, laser_onset + laser_duration)
        
        if laser_start_idx < len(mean_rates) and laser_end_idx <= len(mean_rates):
            return np.trapz(mean_rates[laser_start_idx:laser_end_idx], 
                           bin_centers[laser_start_idx:laser_end_idx])
        return 0

    def plot_psth_comparison(bin_edges, laser_data, control_data, title, save_path,
                            laser_onset=0.5, laser_duration=0.5):
        plt.figure(figsize=(12, 8))
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        
        # laser
        laser_mean, laser_std, laser_trials = laser_data
        lower_bound_laser = np.maximum(laser_mean - laser_std, 0)
        upper_bound_laser = laser_mean + laser_std
        plt.plot(bin_centers, laser_mean, 'r-', label='Laser trials', linewidth=2)
        plt.fill_between(bin_centers, lower_bound_laser, upper_bound_laser, 
                         color='r', alpha=0.2)
        
        # control condition
        control_mean, control_std, control_trials = control_data
        lower_bound_control = np.maximum(control_mean - control_std, 0)
        upper_bound_control = control_mean + control_std
        plt.plot(bin_centers, control_mean, 'b-', label='Control trials', linewidth=2)
        plt.fill_between(bin_centers, lower_bound_control, upper_bound_control, 
                         color='b', alpha=0.2)
        
        # highlight laser period
        plt.axvspan(laser_onset, laser_onset + laser_duration, 
                   color='yellow', alpha=0.2, label='Laser period')
        
        # add sound marker at 0
        plt.axvline(0, color='green', linestyle='--', linewidth=1.5, 
                   label='Sound onset (t=0)')
        
        # calculate and display area difference
        laser_area = calculate_laser_area(bin_centers, laser_mean, laser_onset, laser_duration)
        control_area = calculate_laser_area(bin_centers, control_mean, laser_onset, laser_duration)
        area_diff = laser_area - control_area
        area_percent = (area_diff / control_area * 100) if control_area > 0 else 0
        
        plt.text(0.02, 0.95, 
                f'Area during laser period:\nLaser: {laser_area:.2f}\nControl: {control_area:.2f}\nDiff: {area_diff:.2f} ({area_percent:.1f}%)',
                transform=plt.gca().transAxes, fontsize=10,
                bbox=dict(facecolor='white', alpha=0.8))
        
        # add trial count to the plot
        plt.text(0.02, 0.80, 
                f'Trial counts:\nLaser: {laser_trials}\nControl: {control_trials}',
                transform=plt.gca().transAxes, fontsize=10,
                bbox=dict(facecolor='white', alpha=0.8))
        
        plt.xlabel('Time from sound onset (s)')
        plt.ylabel('Firing Rate (Hz)')
        plt.title(title)
        plt.legend(loc='best')
        plt.ylim(bottom=0)
        plt.grid(alpha=0.3)
        
        # save
        plt.savefig(save_path, format=type_file, dpi=300)
        plt.close()

    # create folders for different conditions
    all_dir = os.path.join(save_folder, 'all')
    reward_dir = os.path.join(save_folder, 'reward')
    nonreward_dir = os.path.join(save_folder, 'nonreward')
    
    for directory in [all_dir, reward_dir, nonreward_dir]:
        os.makedirs(directory, exist_ok=True)
    
    # track processed units for summary
    processed_units = {
        'all': {'right': [], 'left': []},
        'reward': {'right': [], 'left': []},
        'nonreward': {'right': [], 'left': []}
    }
    
    # process each unit
    for unit in units:
        print(f"Processing unit {unit}...")
        
        # get neuron spikes
        neuron_spikes = spikes['sample_index'][spikes['unit_index'] == unit]
        neuron_spikes = neuron_spikes / 40000  # Convert spike times to seconds
        
        # helper function to process each category
        def process_category(category_name, right_laser, left_laser, right_control, left_control, output_dir, 
                             enforce_thresholds=True):
            # process right sounds
            aligned_spikes_right_laser, sc_right_laser = align_spikes_to_sound(
                right_laser, neuron_spikes, time_window)
            bin_edges, mean_rates_right_laser, std_rates_right_laser, n_trials_right_laser = compute_psth(
                aligned_spikes_right_laser, bin_size, time_window)
            
            aligned_spikes_right_control, sc_right_control = align_spikes_to_sound(
                right_control, neuron_spikes, time_window)
            _, mean_rates_right_control, std_rates_right_control, n_trials_right_control = compute_psth(
                aligned_spikes_right_control, bin_size, time_window)
            
            # process left sounds
            aligned_spikes_left_laser, sc_left_laser = align_spikes_to_sound(
                left_laser, neuron_spikes, time_window)
            _, mean_rates_left_laser, std_rates_left_laser, n_trials_left_laser = compute_psth(
                aligned_spikes_left_laser, bin_size, time_window)
            
            aligned_spikes_left_control, sc_left_control = align_spikes_to_sound(
                left_control, neuron_spikes, time_window)
            _, mean_rates_left_control, std_rates_left_control, n_trials_left_control = compute_psth(
                aligned_spikes_left_control, bin_size, time_window)
            
            # threshold for right sounds
            has_enough_right = (n_trials_right_laser >= min_trials and 
                               n_trials_right_control >= min_trials and
                               sc_right_laser >= min_spikes and 
                               sc_right_control >= min_spikes)
            
            # threshold for left sounds
            has_enough_left = (n_trials_left_laser >= min_trials and 
                              n_trials_left_control >= min_trials and
                              sc_left_laser >= min_spikes and 
                              sc_left_control >= min_spikes)
            
            # plot right sounds 
            if has_enough_right and has_enough_left or not enforce_thresholds:
                if n_trials_right_laser > 0 and n_trials_right_control > 0:  # At least have some trials
                    plot_psth_comparison(
                        bin_edges, 
                        (mean_rates_right_laser, std_rates_right_laser, n_trials_right_laser),
                        (mean_rates_right_control, std_rates_right_control, n_trials_right_control),
                        f"Unit {unit} - Right Sound PSTH ({category_name})",
                        os.path.join(output_dir, f"unit_{unit}_right.{type_file}"),
                        laser_onset=laser_delay, 
                        laser_duration=laser_duration
                    )
                    processed_units[category_name]['right'].append(unit)
                    print(f"Unit {unit}: Generated {category_name} right sound PSTH")
                    right_succeeded = True
                else:
                    print(f"Unit {unit}: No trials available for {category_name} right sounds")
                    right_succeeded = False
            else:
                print(f"Unit {unit}: Skipping {category_name} right sounds - insufficient data")
                print(f"  Right laser: {sc_right_laser} spikes, {n_trials_right_laser} trials")
                print(f"  Right control: {sc_right_control} spikes, {n_trials_right_control} trials")
                right_succeeded = False
            
            if has_enough_left and has_enough_right or not enforce_thresholds:
                if n_trials_left_laser > 0 and n_trials_left_control > 0: 
                    plot_psth_comparison(
                        bin_edges, 
                        (mean_rates_left_laser, std_rates_left_laser, n_trials_left_laser),
                        (mean_rates_left_control, std_rates_left_control, n_trials_left_control),
                        f"Unit {unit} - Left Sound PSTH ({category_name})",
                        os.path.join(output_dir, f"unit_{unit}_left.{type_file}"),
                        laser_onset=laser_delay, 
                        laser_duration=laser_duration
                    )
                    processed_units[category_name]['left'].append(unit)
                    print(f"Unit {unit}: Generated {category_name} left sound PSTH")
                    left_succeeded = True
                else:
                    print(f"Unit {unit}: No trials available for {category_name} left sounds")
                    left_succeeded = False
            else:
                print(f"Unit {unit}: Skipping {category_name} left sounds - insufficient data")
                print(f"  Left laser: {sc_left_laser} spikes, {n_trials_left_laser} trials")
                print(f"  Left control: {sc_left_control} spikes, {n_trials_left_control} trials")
                left_succeeded = False
                
            return right_succeeded, left_succeeded, has_enough_right, has_enough_left
        
        all_right, all_left, has_enough_right_all, has_enough_left_all = process_category(
            "all", 
            all_right_laser, all_left_laser, 
            all_right_control, all_left_control,
            all_dir,
            enforce_thresholds=True
        )
        
        if len(reward_right_laser) > 0 or len(reward_left_laser) > 0:
            reward_right, reward_left, _, _ = process_category(
                "reward", 
                reward_right_laser, reward_left_laser, 
                reward_right_control, reward_left_control,
                reward_dir,
                enforce_thresholds=not (has_enough_right_all or has_enough_left_all)
            )
        
        if len(nonreward_right_laser) > 0 or len(nonreward_left_laser) > 0:
            nonreward_right, nonreward_left, _, _ = process_category(
                "nonreward", 
                nonreward_right_laser, nonreward_left_laser, 
                nonreward_right_control, nonreward_left_control,
                nonreward_dir,
                enforce_thresholds=not (has_enough_right_all or has_enough_left_all)
            )
    
    # print summary of processed units
    print("\nProcessing Summary:")
    print(f"Total units: {len(units)}")
    print(f"All category - Right sounds: {len(processed_units['all']['right'])} units")
    print(f"All category - Left sounds: {len(processed_units['all']['left'])} units")
    print(f"Reward category - Right sounds: {len(processed_units['reward']['right'])} units")
    print(f"Reward category - Left sounds: {len(processed_units['reward']['left'])} units")
    print(f"Nonreward category - Right sounds: {len(processed_units['nonreward']['right'])} units")
    print(f"Nonreward category - Left sounds: {len(processed_units['nonreward']['left'])} units")
    

In [5]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
import scipy.io as sio # Keep for loading spikes if still needed, maybe sound times too

def reward_categorized_psth_multipower(
    spikes_path, 
    laser_csv_path, 
    save_folder, 
    target_pulse_width=0.5, 
    pulse_width_tolerance=0.1, 
    palco_val_tolerance=2, 
    csv_path=None, 
    qc_data=None, 
    bin_size=0.005, 
    time_window=(-1, 1.5), 
    type_file='pdf', 
    laser_delay=0.0, 
    laser_duration=0.5, 
    min_trials=3, 
    min_spikes=5, 
    process_individual_conditions=True,
    sound_mat_path=None 
    ):
    """
    Generates PSTH plots categorized by reward outcomes and laser intensity.
    Produces both a combined plot (control vs all lasers) and individual plots 
    (control vs each laser intensity).
    Laser timings, pulse widths, and intensity proxies (PalcoVal) are loaded from a CSV.
    """
    # --- Initial Setup and Data Loading (Identical to previous version) ---
    os.makedirs(save_folder, exist_ok=True)
    spikes = np.load(spikes_path, allow_pickle=True)
    
    print(f"Loading laser data from CSV: {laser_csv_path}")
    try:
        laser_df = pd.read_csv(laser_csv_path)
        required_cols = ['Timestamp', 'PulseWidth', 'PalcoVal']
        if not all(col in laser_df.columns for col in required_cols):
            raise ValueError(f"Laser CSV must contain columns: {required_cols}")
        print(f"Loaded {len(laser_df)} potential laser events.")
        
        pulse_mask = np.abs(laser_df['PulseWidth'] - target_pulse_width) <= pulse_width_tolerance
        valid_pulse_df = laser_df[pulse_mask].copy()
        print(f"Filtered {len(valid_pulse_df)} events based on PulseWidth ({target_pulse_width} +/- {pulse_width_tolerance}s).")
        if len(valid_pulse_df) == 0: raise ValueError("No laser events matching pulse width.")

        palco_05mw, palco_10mw, palco_25mw = 1400, 1300, 990
        valid_pulse_df['Intensity'] = 'Invalid'
        mask_05 = np.abs(valid_pulse_df['PalcoVal'] - palco_05mw) <= palco_val_tolerance
        mask_10 = np.abs(valid_pulse_df['PalcoVal'] - palco_10mw) <= palco_val_tolerance
        mask_25 = np.abs(valid_pulse_df['PalcoVal'] - palco_25mw) <= palco_val_tolerance
        valid_pulse_df.loc[mask_05, 'Intensity'] = '0.5mW'
        valid_pulse_df.loc[mask_10, 'Intensity'] = '1.0mW'
        valid_pulse_df.loc[mask_25, 'Intensity'] = '2.5mW'
        
        invalid_palco_mask = valid_pulse_df['Intensity'] == 'Invalid'
        num_invalid_palco = invalid_palco_mask.sum()
        if num_invalid_palco > 0: print(f"Warning: {num_invalid_palco} events had invalid PalcoVal.")

        laser_times_05mw = valid_pulse_df.loc[valid_pulse_df['Intensity'] == '0.5mW', 'Timestamp'].values
        laser_times_10mw = valid_pulse_df.loc[valid_pulse_df['Intensity'] == '1.0mW', 'Timestamp'].values
        laser_times_25mw = valid_pulse_df.loc[valid_pulse_df['Intensity'] == '2.5mW', 'Timestamp'].values
        print(f"Found intensities: 0.5mW({len(laser_times_05mw)}), 1.0mW({len(laser_times_10mw)}), 2.5mW({len(laser_times_25mw)})")
        
        all_valid_laser_times = valid_pulse_df.loc[~invalid_palco_mask, 'Timestamp'].values
        if len(all_valid_laser_times) == 0: raise ValueError("No valid laser events found after filtering.")
    except FileNotFoundError: raise FileNotFoundError(f"Laser CSV not found: {laser_csv_path}")
    except Exception as e: raise ValueError(f"Error processing laser CSV: {e}")
    
    right_sounds, left_sounds = np.array([]), np.array([])
    if sound_mat_path and os.path.exists(sound_mat_path):
        print(f"Loading sound times from MAT file: {sound_mat_path}")
        # ... (loading logic remains the same)
        try:
            mat_data = sio.loadmat(sound_mat_path)
            if 'right_sounds_evt07' in mat_data and 'left_sounds_evt08' in mat_data:
                 try:
                     right_sounds = mat_data['right_sounds_evt07'][0, 0]['Ts'].flatten()
                     left_sounds = mat_data['left_sounds_evt08'][0, 0]['Ts'].flatten()
                     print(f"Successfully loaded {len(right_sounds)} right sounds and {len(left_sounds)} left sounds from MAT.")
                 except (KeyError, IndexError, AttributeError) as e:
                     print(f"Warning: Error accessing sound data fields in MAT file: {e}")
            else:
                print("Warning: 'right_sounds_evt07' or 'left_sounds_evt08' not found in MAT file.")
        except Exception as e:
            print(f"Warning: Could not load or process sound MAT file: {e}")
    elif csv_path is None: print("Warning: No sound_mat_path or csv_path provided for sound times.")

    # --- Trial Categorization (Identical to previous version) ---
    trial_data_df = None
    if csv_path and os.path.exists(csv_path):
        print(f"Loading trial data from CSV: {csv_path}")
        try:
            trial_data_df = pd.read_csv(csv_path)
            print(f"Loaded {len(trial_data_df)} trials from CSV.")
        except Exception as e: print(f"Warning: Could not load trial CSV: {e}."); trial_data_df = None
    
    # Initialize lists
    reward_right_laser05_times, reward_left_laser05_times = [], []
    nonreward_right_laser05_times, nonreward_left_laser05_times = [], []
    # ... (initialize all other intensity/reward/side lists) ...
    reward_right_laser10_times, reward_left_laser10_times = [], []
    nonreward_right_laser10_times, nonreward_left_laser10_times = [], []
    reward_right_laser25_times, reward_left_laser25_times = [], []
    nonreward_right_laser25_times, nonreward_left_laser25_times = [], []
    reward_right_control_times, reward_left_control_times = [], []
    nonreward_right_control_times, nonreward_left_control_times = [], []


    if trial_data_df is not None:
        print("Categorizing trials using CSV data...")
        laser_times_05mw_sorted = np.sort(laser_times_05mw)
        laser_times_10mw_sorted = np.sort(laser_times_10mw)
        laser_times_25mw_sorted = np.sort(laser_times_25mw)
        
        def find_closest_laser(time, tolerance=1.0):
            # ... (helper function remains the same) ...
            best_match_intensity = None
            min_diff = tolerance 
            for intensity, times_sorted in [('0.5mW', laser_times_05mw_sorted), 
                                           ('1.0mW', laser_times_10mw_sorted),
                                           ('2.5mW', laser_times_25mw_sorted)]:
                if len(times_sorted) == 0: continue
                idx = np.searchsorted(times_sorted, time)
                indices_to_check = []
                if idx > 0: indices_to_check.append(idx - 1) 
                if idx < len(times_sorted): indices_to_check.append(idx) 
                for check_idx in indices_to_check:
                     diff = abs(times_sorted[check_idx] - time)
                     if diff < min_diff:
                         min_diff = diff
                         best_match_intensity = intensity
            return best_match_intensity

        trials_processed, laser_trials_matched, laser_trials_unmatched = 0, 0, 0
        for _, trial in trial_data_df.iterrows(): 
            # ... (trial processing loop remains the same, populating all lists) ...
            trials_processed += 1
            try:
                trial_time = trial['TimeStart']; is_laser_trial = trial['IsLaserTrial'] == 1
                is_right = trial['TrialSide'] == 'Right'; is_rewarded = trial['RMI'] == 'reward'
                if is_laser_trial:
                    matched_intensity = find_closest_laser(trial_time)
                    if matched_intensity:
                        laser_trials_matched += 1
                        # Append to appropriate list based on intensity, side, reward
                        if is_right:
                            target_list_reward = None; target_list_nonreward = None
                            if matched_intensity == '0.5mW': target_list_reward, target_list_nonreward = reward_right_laser05_times, nonreward_right_laser05_times
                            elif matched_intensity == '1.0mW': target_list_reward, target_list_nonreward = reward_right_laser10_times, nonreward_right_laser10_times
                            elif matched_intensity == '2.5mW': target_list_reward, target_list_nonreward = reward_right_laser25_times, nonreward_right_laser25_times
                            if is_rewarded: target_list_reward.append(trial_time)
                            else: target_list_nonreward.append(trial_time)
                        else: # Left
                            target_list_reward = None; target_list_nonreward = None
                            if matched_intensity == '0.5mW': target_list_reward, target_list_nonreward = reward_left_laser05_times, nonreward_left_laser05_times
                            elif matched_intensity == '1.0mW': target_list_reward, target_list_nonreward = reward_left_laser10_times, nonreward_left_laser10_times
                            elif matched_intensity == '2.5mW': target_list_reward, target_list_nonreward = reward_left_laser25_times, nonreward_left_laser25_times
                            if is_rewarded: target_list_reward.append(trial_time)
                            else: target_list_nonreward.append(trial_time)
                    else: laser_trials_unmatched += 1
                else: # Control
                    if is_right:
                        if is_rewarded: reward_right_control_times.append(trial_time)
                        else: nonreward_right_control_times.append(trial_time)
                    else: # Left
                        if is_rewarded: reward_left_control_times.append(trial_time)
                        else: nonreward_left_control_times.append(trial_time)
            except KeyError as e: print(f"Error trial {trials_processed}: Missing key {e}. Skip."); continue
            except Exception as e: print(f"Error trial {trials_processed}: {e}. Skip."); continue

        print(f"Finished processing {trials_processed} trials.")
        if laser_trials_unmatched > 0: print(f"Warning: {laser_trials_unmatched} unmatched laser trials.")

        # Convert lists to numpy arrays
        reward_right_laser05 = np.array(reward_right_laser05_times) 
        # ... (convert all other lists) ...
        reward_left_laser05 = np.array(reward_left_laser05_times)
        nonreward_right_laser05 = np.array(nonreward_right_laser05_times)
        nonreward_left_laser05 = np.array(nonreward_left_laser05_times)
        reward_right_laser10 = np.array(reward_right_laser10_times)
        reward_left_laser10 = np.array(reward_left_laser10_times)
        nonreward_right_laser10 = np.array(nonreward_right_laser10_times)
        nonreward_left_laser10 = np.array(nonreward_left_laser10_times)
        reward_right_laser25 = np.array(reward_right_laser25_times)
        reward_left_laser25 = np.array(reward_left_laser25_times)
        nonreward_right_laser25 = np.array(nonreward_right_laser25_times)
        nonreward_left_laser25 = np.array(nonreward_left_laser25_times)
        reward_right_control = np.array(reward_right_control_times)
        reward_left_control = np.array(reward_left_control_times)
        nonreward_right_control = np.array(nonreward_right_control_times)
        nonreward_left_control = np.array(nonreward_left_control_times)
        
        # Combine for 'all' category
        all_right_laser05 = np.concatenate([reward_right_laser05, nonreward_right_laser05])
        # ... (concatenate all others) ...
        all_left_laser05 = np.concatenate([reward_left_laser05, nonreward_left_laser05])
        all_right_laser10 = np.concatenate([reward_right_laser10, nonreward_right_laser10])
        all_left_laser10 = np.concatenate([reward_left_laser10, nonreward_left_laser10])
        all_right_laser25 = np.concatenate([reward_right_laser25, nonreward_right_laser25])
        all_left_laser25 = np.concatenate([reward_left_laser25, nonreward_left_laser25])
        all_right_control = np.concatenate([reward_right_control, nonreward_right_control])
        all_left_control = np.concatenate([reward_left_control, nonreward_left_control])

        # Print trial counts summary (remains the same)
        print(f"\nTrial categorization summary:")
        print(f"  Control Right: Reward={len(reward_right_control)}, NonReward={len(nonreward_right_control)}, Total={len(all_right_control)}")
        # ... (print summary for all categories) ...
        print(f"  Control Left:  Reward={len(reward_left_control)}, NonReward={len(nonreward_left_control)}, Total={len(all_left_control)}")
        print(f"  Laser 0.5mW Right: Reward={len(reward_right_laser05)}, NonReward={len(nonreward_right_laser05)}, Total={len(all_right_laser05)}")
        print(f"  Laser 0.5mW Left:  Reward={len(reward_left_laser05)}, NonReward={len(nonreward_left_laser05)}, Total={len(all_left_laser05)}")
        print(f"  Laser 1.0mW Right: Reward={len(reward_right_laser10)}, NonReward={len(nonreward_right_laser10)}, Total={len(all_right_laser10)}")
        print(f"  Laser 1.0mW Left:  Reward={len(reward_left_laser10)}, NonReward={len(nonreward_left_laser10)}, Total={len(all_left_laser10)}")
        print(f"  Laser 2.5mW Right: Reward={len(reward_right_laser25)}, NonReward={len(nonreward_right_laser25)}, Total={len(all_right_laser25)}")
        print(f"  Laser 2.5mW Left:  Reward={len(reward_left_laser25)}, NonReward={len(nonreward_left_laser25)}, Total={len(all_left_laser25)}")


    else: # Fallback: No Trial CSV Data
        print("No trial data CSV. Using sound event times for basic categorization.")
        if len(right_sounds) == 0 and len(left_sounds) == 0: raise ValueError("No sound times and no trial CSV.")
        # ... (Fallback logic remains the same, populating 'all_*' arrays based on sound proximity) ...
        def find_closest_laser(time, tolerance=1.0): # Re-define if needed in this scope
            best_match_intensity = None; min_diff = tolerance 
            for intensity, times_sorted in [('0.5mW', np.sort(laser_times_05mw)), 
                                           ('1.0mW', np.sort(laser_times_10mw)),
                                           ('2.5mW', np.sort(laser_times_25mw))]:
                if len(times_sorted) == 0: continue
                idx = np.searchsorted(times_sorted, time)
                indices_to_check = []
                if idx > 0: indices_to_check.append(idx - 1) 
                if idx < len(times_sorted): indices_to_check.append(idx) 
                for check_idx in indices_to_check:
                     diff = abs(times_sorted[check_idx] - time)
                     if diff < min_diff: min_diff = diff; best_match_intensity = intensity
            return best_match_intensity
            
        all_right_laser05, all_left_laser05 = [], []
        all_right_laser10, all_left_laser10 = [], []
        all_right_laser25, all_left_laser25 = [], []
        all_right_control, all_left_control = [], []
        processed_sound_times = set()
        for sound_time in right_sounds:
             if sound_time in processed_sound_times: continue
             intensity = find_closest_laser(sound_time); processed_sound_times.add(sound_time)
             if intensity == '0.5mW': all_right_laser05.append(sound_time)
             elif intensity == '1.0mW': all_right_laser10.append(sound_time)
             elif intensity == '2.5mW': all_right_laser25.append(sound_time)
             else: all_right_control.append(sound_time)
        for sound_time in left_sounds:
             if sound_time in processed_sound_times: continue
             intensity = find_closest_laser(sound_time); processed_sound_times.add(sound_time)
             if intensity == '0.5mW': all_left_laser05.append(sound_time)
             elif intensity == '1.0mW': all_left_laser10.append(sound_time)
             elif intensity == '2.5mW': all_left_laser25.append(sound_time)
             else: all_left_control.append(sound_time)
             
        all_right_laser05 = np.array(all_right_laser05); all_left_laser05 = np.array(all_left_laser05)
        all_right_laser10 = np.array(all_right_laser10); all_left_laser10 = np.array(all_left_laser10)
        all_right_laser25 = np.array(all_right_laser25); all_left_laser25 = np.array(all_left_laser25)
        all_right_control = np.array(all_right_control); all_left_control = np.array(all_left_control)
        # Reward categories remain empty
        reward_right_laser05, reward_left_laser05 = np.array([]), np.array([])
        nonreward_right_laser05, nonreward_left_laser05 = np.array([]), np.array([])
        reward_right_laser10, reward_left_laser10 = np.array([]), np.array([])
        nonreward_right_laser10, nonreward_left_laser10 = np.array([]), np.array([])
        reward_right_laser25, reward_left_laser25 = np.array([]), np.array([])
        nonreward_right_laser25, nonreward_left_laser25 = np.array([]), np.array([])
        reward_right_control, reward_left_control = np.array([]), np.array([])
        nonreward_right_control, nonreward_left_control = np.array([]), np.array([])

        print(f"\nTrial counts (estimated from sound proximity):")
        # ... (print summary for fallback) ...
        print(f"  Control Right: {len(all_right_control)}")
        print(f"  Control Left:  {len(all_left_control)}")
        print(f"  Laser 0.5mW Right: {len(all_right_laser05)}")
        print(f"  Laser 0.5mW Left:  {len(all_left_laser05)}")
        print(f"  Laser 1.0mW Right: {len(all_right_laser10)}")
        print(f"  Laser 1.0mW Left:  {len(all_left_laser10)}")
        print(f"  Laser 2.5mW Right: {len(all_right_laser25)}")
        print(f"  Laser 2.5mW Left:  {len(all_left_laser25)}")
        print("  (Reward/Nonreward categorization not available)")

    # --- Unit Processing Setup (Identical) ---
    if qc_data is not None:
        try: qc = pd.read_csv(qc_data); units = qc.iloc[:, 0].values; print(f"Analyzing {len(units)} QC units.")
        except Exception as e: print(f"Warning: QC load error: {e}. Analyzing all units."); units = np.unique(spikes['unit_index'])
    else: units = np.unique(spikes['unit_index']); print(f"Analyzing all {len(units)} units.")

    # --- Helper Functions (Identical) ---
    def align_spikes_to_sound(sound_times, neuron_spikes, time_window):
        # ... (remains the same) ...
        if len(sound_times) == 0: return [], 0
        aligned_spikes = []; spikes_count = 0
        for st in sound_times:
            spike_window = neuron_spikes[(neuron_spikes >= st + time_window[0]) & (neuron_spikes <= st + time_window[1])]
            spikes_count += len(spike_window); aligned_spikes.append(spike_window - st)
        return aligned_spikes, spikes_count

    def compute_psth(aligned_spikes, bin_size, time_window):
        # ... (remains the same) ...
        bin_edges = np.arange(time_window[0], time_window[1] + bin_size, bin_size); num_bins = len(bin_edges) - 1
        num_trials = len(aligned_spikes); 
        if num_trials == 0: return bin_edges, np.zeros(num_bins), np.zeros(num_bins), 0
        spike_counts = np.zeros((num_trials, num_bins))
        for i, trial_spikes in enumerate(aligned_spikes): counts, _ = np.histogram(trial_spikes, bins=bin_edges); spike_counts[i, :] = counts
        firing_rates = spike_counts / bin_size; mean_rates = np.mean(firing_rates, axis=0)
        std_rates = np.std(firing_rates, axis=0) / np.sqrt(num_trials) if num_trials > 1 else np.zeros(num_bins)
        return bin_edges, mean_rates, std_rates, num_trials

    def calculate_laser_area(bin_centers, mean_rates, laser_onset=0.8, laser_duration=0.5):
        # ... (remains the same) ...
         laser_start_idx = np.searchsorted(bin_centers, laser_onset, side='left')
         laser_end_idx = np.searchsorted(bin_centers, laser_onset + laser_duration, side='right')
         if laser_start_idx < len(mean_rates) and laser_end_idx <= len(mean_rates) and laser_start_idx < laser_end_idx:
             return np.trapz(mean_rates[laser_start_idx:laser_end_idx], bin_centers[laser_start_idx:laser_end_idx])
         elif laser_start_idx == laser_end_idx and laser_start_idx < len(mean_rates):
              bin_width = bin_centers[1] - bin_centers[0] if len(bin_centers) > 1 else laser_duration
              effective_duration_in_bin = min(laser_duration, bin_width)
              return mean_rates[laser_start_idx] * effective_duration_in_bin
         return 0.0

    # --- Plotting Function for Combined Plot (Identical) ---
    def plot_psth_multi_laser(bin_edges, control_data, laser_data_dict, title, save_path, laser_onset=0.8, laser_duration=0.5):
        # ... (remains the same as previous version) ...
        plt.figure(figsize=(14, 8)); bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2
        colors = {'0.5mW': 'red', '1.0mW': 'magenta', '2.5mW': 'orange'}; control_color = 'blue'
        max_rate = 0
        control_mean, control_std, control_trials = control_data
        if control_trials > 0:
            lower_bound_control = np.maximum(control_mean - control_std, 0); upper_bound_control = control_mean + control_std
            plt.plot(bin_centers, control_mean, color=control_color, linestyle='-', label=f'Control (n={control_trials})', linewidth=2)
            plt.fill_between(bin_centers, lower_bound_control, upper_bound_control, color=control_color, alpha=0.15)
            max_rate = max(max_rate, np.max(upper_bound_control) if len(upper_bound_control) > 0 else 0)
            control_area = calculate_laser_area(bin_centers, control_mean, laser_onset, laser_duration)
        else: control_area = 0; plt.plot([], [], color=control_color, linestyle='-', label='Control (n=0)', linewidth=2)
        area_text = f'Area ({laser_onset:.2f}-{laser_onset+laser_duration:.2f}s):\nControl: {control_area:.2f} (n={control_trials})'
        for intensity, (laser_mean, laser_std, laser_trials) in laser_data_dict.items():
            if laser_trials > 0:
                 color = colors.get(intensity, 'gray')
                 lower_bound_laser = np.maximum(laser_mean - laser_std, 0); upper_bound_laser = laser_mean + laser_std
                 plt.plot(bin_centers, laser_mean, color=color, linestyle='-', label=f'Laser {intensity} (n={laser_trials})', linewidth=1.5)
                 plt.fill_between(bin_centers, lower_bound_laser, upper_bound_laser, color=color, alpha=0.15)
                 max_rate = max(max_rate, np.max(upper_bound_laser) if len(upper_bound_laser) > 0 else 0)
                 laser_area = calculate_laser_area(bin_centers, laser_mean, laser_onset, laser_duration)
                 area_diff = laser_area - control_area
                 area_percent = (area_diff / control_area * 100) if control_area > 1e-9 else (np.inf if area_diff > 1e-9 else 0)
                 area_text += f'\n{intensity}: {laser_area:.2f} (n={laser_trials}), Diff: {area_diff:.2f} ({area_percent:.1f}%)'
            else: plt.plot([],[], color=colors.get(intensity, 'gray'), linestyle='-', label=f'Laser {intensity} (n=0)', linewidth=1.5); area_text += f'\n{intensity}: N/A (n=0)'
        plt.axvspan(laser_onset, laser_onset + laser_duration, color='yellow', alpha=0.2, label='Laser period')
        plt.axvline(0, color='green', linestyle='--', linewidth=1.5, label='Sound onset (t=0)')
        plt.text(0.02, 0.98, area_text, transform=plt.gca().transAxes, fontsize=9, va='top', bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
        plt.xlabel('Time from sound onset (s)'); plt.ylabel('Firing Rate (Hz)'); plt.title(title, fontsize=12)
        plt.legend(loc='upper right', fontsize=9); plt.ylim(bottom=0, top=max(1, max_rate * 1.1)); plt.grid(alpha=0.3)
        plt.savefig(save_path, format=type_file, dpi=300, bbox_inches='tight'); plt.close()


    # --- *** NEW: Plotting Function for Single Laser vs Control Comparison *** ---
    def plot_psth_single_comparison(bin_edges, control_data, laser_data, laser_intensity, title, save_path, 
                                    laser_onset=0.0, laser_duration=0.5):
        plt.figure(figsize=(12, 8)) # Standard size for single comparison
        bin_centers = bin_edges[:-1] + np.diff(bin_edges) / 2

        colors = {'0.5mW': 'orange', '1.0mW': 'magenta', '2.5mW': 'red'}
        control_color = 'blue'
        laser_color = colors.get(laser_intensity, 'red') # Default to red if intensity name mismatch

        max_rate = 0

        # Control condition
        control_mean, control_std, control_trials = control_data
        if control_trials > 0:
            lower_bound_control = np.maximum(control_mean - control_std, 0)
            upper_bound_control = control_mean + control_std
            plt.plot(bin_centers, control_mean, color=control_color, linestyle='-', label=f'Control (n={control_trials})', linewidth=2)
            plt.fill_between(bin_centers, lower_bound_control, upper_bound_control, color=control_color, alpha=0.2)
            max_rate = max(max_rate, np.max(upper_bound_control) if len(upper_bound_control) > 0 else 0)
            control_area = calculate_laser_area(bin_centers, control_mean, laser_onset, laser_duration)
        else:
            control_area = 0
            plt.plot([], [], color=control_color, linestyle='-', label='Control (n=0)', linewidth=2) 

        # Single Laser condition
        laser_mean, laser_std, laser_trials = laser_data
        if laser_trials > 0:
            lower_bound_laser = np.maximum(laser_mean - laser_std, 0)
            upper_bound_laser = laser_mean + laser_std
            plt.plot(bin_centers, laser_mean, color=laser_color, linestyle='-', label=f'Laser {laser_intensity} (n={laser_trials})', linewidth=2)
            plt.fill_between(bin_centers, lower_bound_laser, upper_bound_laser, color=laser_color, alpha=0.2)
            max_rate = max(max_rate, np.max(upper_bound_laser) if len(upper_bound_laser) > 0 else 0)
            laser_area = calculate_laser_area(bin_centers, laser_mean, laser_onset, laser_duration)
            area_diff = laser_area - control_area
            area_percent = (area_diff / control_area * 100) if control_area > 1e-9 else (np.inf if area_diff > 1e-9 else 0)
            area_text = (f'Area ({laser_onset:.2f}-{laser_onset+laser_duration:.2f}s):\n'
                         f'Control: {control_area:.2f} (n={control_trials})\n'
                         f'{laser_intensity}: {laser_area:.2f} (n={laser_trials})\n'
                         f'Diff: {area_diff:.2f} ({area_percent:.1f}%)')
        else:
            laser_area = 0
            plt.plot([],[], color=laser_color, linestyle='-', label=f'Laser {laser_intensity} (n=0)', linewidth=2)
            area_text = (f'Area ({laser_onset:.2f}-{laser_onset+laser_duration:.2f}s):\n'
                         f'Control: {control_area:.2f} (n={control_trials})\n'
                         f'{laser_intensity}: N/A (n=0)')

        # Highlight laser period and sound onset
        plt.axvspan(laser_onset, laser_onset + laser_duration, color='yellow', alpha=0.2, label='Laser period')
        plt.axvline(0, color='green', linestyle='--', linewidth=1.5, label='Sound onset (t=0)')

        # Add text box
        plt.text(0.02, 0.98, area_text, transform=plt.gca().transAxes, fontsize=10, va='top', 
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

        plt.xlabel('Time from sound onset (s)'); plt.ylabel('Firing Rate (Hz)'); plt.title(title, fontsize=12)
        plt.legend(loc='upper right', fontsize=10); plt.ylim(bottom=0, top=max(1, max_rate * 1.1)); plt.grid(alpha=0.3)
        plt.savefig(save_path, format=type_file, dpi=300, bbox_inches='tight'); plt.close()


    # --- Modified Processing Loop ---
    all_dir = os.path.join(save_folder, 'all')
    reward_dir = os.path.join(save_folder, 'reward')
    nonreward_dir = os.path.join(save_folder, 'nonreward')
    for directory in [all_dir, reward_dir, nonreward_dir]: os.makedirs(directory, exist_ok=True)
        
    processed_units = {'all': {'right': 0, 'left': 0}, 'reward': {'right': 0, 'left': 0}, 'nonreward': {'right': 0, 'left': 0}}
    
    # Modified helper function to generate both combined and individual plots
    def process_category(category_name, 
                         right_laser05_times, left_laser05_times,
                         right_laser10_times, left_laser10_times,
                         right_laser25_times, left_laser25_times,
                         right_control_times, left_control_times, 
                         output_dir):
        
        nonlocal neuron_spikes, bin_size, time_window, min_trials, min_spikes, type_file, laser_delay, laser_duration, unit
                         
        # --- Process RIGHT sounds ---
        aligned_spikes_right_control, sc_right_control = align_spikes_to_sound(right_control_times, neuron_spikes, time_window)
        bin_edges, mean_rates_right_control, std_rates_right_control, n_trials_right_control = compute_psth(aligned_spikes_right_control, bin_size, time_window)
        
        right_laser_psth_data = {} 
        right_plotted_combined = False # Track if combined plot was made
        right_plotted_individual_count = 0 # Track how many individual plots were made

        # Calculate PSTH for all laser intensities first
        for intensity, times in [('0.5mW', right_laser05_times), ('1.0mW', right_laser10_times), ('2.5mW', right_laser25_times)]:
            aligned_spikes, sc_laser = align_spikes_to_sound(times, neuron_spikes, time_window)
            _, mean_rates, std_rates, n_trials = compute_psth(aligned_spikes, bin_size, time_window)
            right_laser_psth_data[intensity] = (mean_rates, std_rates, n_trials) # Store results

        # Check if *any* laser intensity + control has sufficient data for the combined plot
        has_sufficient_data_for_combined_right = any(
            psth_data[2] >= min_trials and n_trials_right_control >= min_trials and
            align_spikes_to_sound(times, neuron_spikes, time_window)[1] >= min_spikes and sc_right_control >= min_spikes
            for intensity, times in [('0.5mW', right_laser05_times), ('1.0mW', right_laser10_times), ('2.5mW', right_laser25_times)]
            if (psth_data := right_laser_psth_data[intensity]) is not None # Use walrus operator (Python 3.8+)
        )
        
        # Generate Combined Plot for Right sounds (if criteria met)
        if n_trials_right_control > 0 and has_sufficient_data_for_combined_right:
            plot_psth_multi_laser(
                bin_edges, 
                (mean_rates_right_control, std_rates_right_control, n_trials_right_control),
                right_laser_psth_data,
                f"Unit {unit} - Right Sound PSTH ({category_name} - Combined)", # Modified title
                os.path.join(output_dir, f"unit_{unit}_right_combined.{type_file}"), # Modified filename
                laser_onset=laser_delay, laser_duration=laser_duration
            )
            print(f"  Unit {unit}: Generated COMBINED {category_name} right sound plot.")
            right_plotted_combined = True
            # Increment overall count only once per side/category, even if multiple plots generated
            processed_units[category_name]['right'] += 1 
        elif n_trials_right_control > 0 and any(d[2] > 0 for d in right_laser_psth_data.values()):
             print(f"  Unit {unit}: Skipping COMBINED {category_name} right plot - insufficient trials/spikes (ControlN={n_trials_right_control}, LaserNs={[d[2] for d in right_laser_psth_data.values()]}).")
        else:
             print(f"  Unit {unit}: Skipping COMBINED {category_name} right plot - no trials for control or all laser conditions.")

        # *** NEW: Generate Individual Plots for Right Sounds ***
        for intensity, (mean_rates, std_rates, n_trials) in right_laser_psth_data.items():
            # Check criteria for this specific intensity vs control
            aligned_spikes_laser, sc_laser = align_spikes_to_sound(locals()[f'right_laser{intensity.replace("mW","").replace(".","")}_times'], neuron_spikes, time_window) # Get original times list
            
            if (n_trials >= min_trials and n_trials_right_control >= min_trials and
                sc_laser >= min_spikes and sc_right_control >= min_spikes):
                
                # Construct filename and title for individual plot
                intensity_tag = intensity.replace('.', 'p') # e.g., 0p5mW for filename
                individual_save_path = os.path.join(output_dir, f"unit_{unit}_right_vs_{intensity_tag}.{type_file}")
                individual_title = f"Unit {unit} - Right Sound PSTH ({category_name} - Control vs {intensity})"

                plot_psth_single_comparison(
                    bin_edges,
                    (mean_rates_right_control, std_rates_right_control, n_trials_right_control), # Control data
                    (mean_rates, std_rates, n_trials), # Current laser intensity data
                    intensity, # Pass intensity name for labeling
                    individual_title,
                    individual_save_path,
                    laser_onset=laser_delay, laser_duration=laser_duration
                )
                print(f"  Unit {unit}: Generated INDIVIDUAL {category_name} right sound plot (vs {intensity}).")
                right_plotted_individual_count += 1
            # Optional: Add an else here to print why an individual plot was skipped if needed

        # --- Process LEFT sounds (apply the same logic) ---
        aligned_spikes_left_control, sc_left_control = align_spikes_to_sound(left_control_times, neuron_spikes, time_window)
        bin_edges, mean_rates_left_control, std_rates_left_control, n_trials_left_control = compute_psth(aligned_spikes_left_control, bin_size, time_window)

        left_laser_psth_data = {}
        left_plotted_combined = False
        left_plotted_individual_count = 0

        # Calculate all left laser PSTHs
        for intensity, times in [('0.5mW', left_laser05_times), ('1.0mW', left_laser10_times), ('2.5mW', left_laser25_times)]:
            aligned_spikes, sc_laser = align_spikes_to_sound(times, neuron_spikes, time_window)
            _, mean_rates, std_rates, n_trials = compute_psth(aligned_spikes, bin_size, time_window)
            left_laser_psth_data[intensity] = (mean_rates, std_rates, n_trials)

        # Check criteria for combined left plot
        has_sufficient_data_for_combined_left = any(
            psth_data[2] >= min_trials and n_trials_left_control >= min_trials and
            align_spikes_to_sound(times, neuron_spikes, time_window)[1] >= min_spikes and sc_left_control >= min_spikes
            for intensity, times in [('0.5mW', left_laser05_times), ('1.0mW', left_laser10_times), ('2.5mW', left_laser25_times)]
            if (psth_data := left_laser_psth_data[intensity]) is not None
        )
        
        # Generate Combined Plot for Left sounds
        if n_trials_left_control > 0 and has_sufficient_data_for_combined_left:
            plot_psth_multi_laser(
                bin_edges, 
                (mean_rates_left_control, std_rates_left_control, n_trials_left_control),
                left_laser_psth_data,
                f"Unit {unit} - Left Sound PSTH ({category_name} - Combined)",
                os.path.join(output_dir, f"unit_{unit}_left_combined.{type_file}"),
                laser_onset=laser_delay, laser_duration=laser_duration
            )
            print(f"  Unit {unit}: Generated COMBINED {category_name} left sound plot.")
            left_plotted_combined = True
            processed_units[category_name]['left'] += 1 # Increment overall count
        elif n_trials_left_control > 0 and any(d[2] > 0 for d in left_laser_psth_data.values()):
             print(f"  Unit {unit}: Skipping COMBINED {category_name} left plot - insufficient trials/spikes (ControlN={n_trials_left_control}, LaserNs={[d[2] for d in left_laser_psth_data.values()]}).")
        else:
            print(f"  Unit {unit}: Skipping COMBINED {category_name} left plot - no trials for control or all laser conditions.")

        # *** NEW: Generate Individual Plots for Left Sounds ***
        for intensity, (mean_rates, std_rates, n_trials) in left_laser_psth_data.items():
            aligned_spikes_laser, sc_laser = align_spikes_to_sound(locals()[f'left_laser{intensity.replace("mW","").replace(".","")}_times'], neuron_spikes, time_window)
            
            if (n_trials >= min_trials and n_trials_left_control >= min_trials and
                sc_laser >= min_spikes and sc_left_control >= min_spikes):
                
                intensity_tag = intensity.replace('.', 'p')
                individual_save_path = os.path.join(output_dir, f"unit_{unit}_left_vs_{intensity_tag}.{type_file}")
                individual_title = f"Unit {unit} - Left Sound PSTH ({category_name} - Control vs {intensity})"

                plot_psth_single_comparison(
                    bin_edges,
                    (mean_rates_left_control, std_rates_left_control, n_trials_left_control),
                    (mean_rates, std_rates, n_trials), 
                    intensity,
                    individual_title,
                    individual_save_path,
                    laser_onset=laser_delay, laser_duration=laser_duration
                )
                print(f"  Unit {unit}: Generated INDIVIDUAL {category_name} left sound plot (vs {intensity}).")
                left_plotted_individual_count += 1
                
        # Return value indicates if *any* plot (combined or individual) was generated for that side
        return right_plotted_combined or (right_plotted_individual_count > 0), \
               left_plotted_combined or (left_plotted_individual_count > 0)

    # Process each unit
    for unit_idx, unit in enumerate(units):
        print(f"\nProcessing unit {unit} ({unit_idx+1}/{len(units)})...")
        neuron_spike_indices = spikes['sample_index'][spikes['unit_index'] == unit]
        sampling_rate = 40000 # Adjust if needed
        neuron_spikes = neuron_spike_indices / sampling_rate 
        if len(neuron_spikes) == 0: print(f"  Unit {unit}: No spikes. Skip."); continue

        # Process 'all' category
        process_category(
            "all", 
            all_right_laser05, all_left_laser05, all_right_laser10, all_left_laser10, all_right_laser25, all_left_laser25,
            all_right_control, all_left_control, all_dir
        )
        
        # Process 'reward' and 'nonreward' if applicable
        if trial_data_df is not None and process_individual_conditions:
            process_category(
                "reward", 
                reward_right_laser05, reward_left_laser05, reward_right_laser10, reward_left_laser10, reward_right_laser25, reward_left_laser25,
                reward_right_control, reward_left_control, reward_dir
            )
            process_category(
                "nonreward", 
                nonreward_right_laser05, nonreward_left_laser05, nonreward_right_laser10, nonreward_left_laser10, nonreward_right_laser25, nonreward_left_laser25,
                nonreward_right_control, nonreward_left_control, nonreward_dir
            )
        elif process_individual_conditions: print(f"  Unit {unit}: Skip reward/nonreward (no trial CSV).")

    # Print summary (remains the same - counts units for which *any* plot was made)
    print("\n--- Processing Summary ---")
    print(f"Total unique units analyzed: {len(units)}")
    print(f"Units with plots generated for 'All' category - Right sounds: {processed_units['all']['right']}")
    print(f"Units with plots generated for 'All' category - Left sounds:  {processed_units['all']['left']}")
    if process_individual_conditions and trial_data_df is not None:
        print(f"Units with plots generated for 'Reward' category - Right sounds: {processed_units['reward']['right']}")
        print(f"Units with plots generated for 'Reward' category - Left sounds:  {processed_units['reward']['left']}")
        print(f"Units with plots generated for 'Nonreward' category - Right sounds: {processed_units['nonreward']['right']}")
        print(f"Units with plots generated for 'Nonreward' category - Left sounds:  {processed_units['nonreward']['left']}")
    print("--------------------------")
    print("Note: Counts indicate units for which at least one plot (combined or individual) was generated per category/side.")


Multipower analysis

In [6]:
session_id = "Rec_Upstream_DCN_1_250408_MixedmW_500ms_0delay_040825001"
base_folder = rf"E:\Paolo\temp_recordings\DCN_1\{session_id}"
bin_size = 0.02
output_folder_psth_1 = "laser_psth_20bin_500laser_0delay"
output_folder_psth_1_path = rf"{base_folder}\spikeinterface\{output_folder_psth_1}"
spikes_path = rf"{base_folder}\spikeinterface\analyzer\sorting\spikes.npy"
laser_csv_path = rf"{base_folder}\pulses.csv"
reward_trials_path = rf"{base_folder}\{session_id}_analysis.mat"
csv_path_csv = rf"{base_folder}\trial_data.csv"
full_trial_data_df = pd.read_csv(csv_path_csv)

reward_categorized_psth_multipower(spikes_path=spikes_path, laser_csv_path=laser_csv_path,save_folder=output_folder_psth_1_path, 
    csv_path=csv_path_csv, bin_size=bin_size)

Loading laser data from CSV: E:\Paolo\temp_recordings\DCN_1\Rec_Upstream_DCN_1_250408_MixedmW_500ms_0delay_040825001\pulses.csv
Loaded 119 potential laser events.
Filtered 117 events based on PulseWidth (0.5 +/- 0.1s).
Found intensities: 0.5mW(44), 1.0mW(35), 2.5mW(38)
Loading trial data from CSV: E:\Paolo\temp_recordings\DCN_1\Rec_Upstream_DCN_1_250408_MixedmW_500ms_0delay_040825001\trial_data.csv
Loaded 416 trials from CSV.
Categorizing trials using CSV data...
Finished processing 416 trials.

Trial categorization summary:
  Control Right: Reward=103, NonReward=49, Total=152
  Control Left:  Reward=144, NonReward=3, Total=147
  Laser 0.5mW Right: Reward=30, NonReward=0, Total=30
  Laser 0.5mW Left:  Reward=10, NonReward=3, Total=13
  Laser 1.0mW Right: Reward=11, NonReward=2, Total=13
  Laser 1.0mW Left:  Reward=11, NonReward=11, Total=22
  Laser 2.5mW Right: Reward=0, NonReward=19, Total=19
  Laser 2.5mW Left:  Reward=0, NonReward=19, Total=19
Analyzing all 194 units.

Processing un

Optotag Recording Script - DCN

In [14]:
########################## VARIABLES TO CHANGE ###########################
session_id = "Rec_Upstream_DCN_1_250405_2point5mW_50msplus50ramp_040525001"
base_folder = rf"N:\MICROSCOPE\Paolo\Recordings\Rec_Upstream_DCN\Rec_Upstream_DCN_1_SI\{session_id}"

bin_size = 0.02
laser_delay_1 = 0.8
laser_duration = 0.1 # 100ms
output_folder_psth_1 = "laser_psth_20bin_500laser_800delay"

# if split trial
split_trial = True
laser_delay_2 = 0 # 0ms

block_1_start = 1
block_1_end = 220 # end of block 1
block_2_start = 224
block_2_end = 445 # end of block 2

output_folder_psth_2 = "laser_psth_20bin_500laser_0delay"

####################### DO NOT CHANGE BELOW ########################

output_folder_psth_1_path = rf"{base_folder}\spikeinterface\{output_folder_psth_1}"
output_folder_psth_2_path = rf"{base_folder}\spikeinterface\{output_folder_psth_2}"
spikes_path = rf"{base_folder}\spikeinterface\analyzer\sorting\spikes.npy"
laser_timestamps_path = rf"{base_folder}\{session_id}.mat"
reward_trials_path = rf"{base_folder}\{session_id}_analysis.mat"
laser_type = "laser_on_evt05"

#of laser, for of comparison later
of_laser = "laser_on_evt14"

csv_path_csv = rf"{base_folder}\trial_data.csv"
full_trial_data_df = pd.read_csv(csv_path_csv)

trials_block1_df = full_trial_data_df.iloc[block_1_start:block_1_end].copy()
trials_block2_df = full_trial_data_df.iloc[block_2_start:block_2_end].copy()

csv_path = full_trial_data_df

# if not split trial
if not split_trial:
    reward_categorized_psth(spikes_path=spikes_path,laser_times_path=laser_timestamps_path,
                            save_folder=output_folder_psth_1_path, csv_path=csv_path,
                            laser_type=laser_type,bin_size=bin_size,laser_delay=laser_delay_1,
                            laser_duration=laser_duration)

# if split trial
if split_trial:
    # first block
    reward_categorized_psth(spikes_path=spikes_path,laser_times_path=laser_timestamps_path,
                            save_folder=output_folder_psth_1_path, csv_path=trials_block1_df,
                            laser_type=laser_type,bin_size=bin_size,laser_delay=laser_delay_1,
                            laser_duration=laser_duration)
    # second block
    reward_categorized_psth(spikes_path=spikes_path,laser_times_path=laser_timestamps_path,
                            save_folder=output_folder_psth_2_path, csv_path=trials_block2_df,
                            laser_type=laser_type,bin_size=bin_size,laser_delay=laser_delay_2,
                            laser_duration=laser_duration)

#evt_laser = 'evt_timestamps'

Categorizing trials using CSV data...
Trial categorization summary:
  Right laser trials: 28 total
    Reward: 20
    Non-reward: 8
  Left laser trials: 30 total
    Reward: 28
    Non-reward: 2
  Right control trials: 79 total
    Reward: 45
    Non-reward: 34
  Left control trials: 82 total
    Reward: 75
    Non-reward: 7
Successfully loaded 128 laser timestamps
Successfully loaded 220 right sounds and 222 left sounds
Processing unit 0...
Unit 0: Generated all right sound PSTH
Unit 0: Generated all left sound PSTH
Unit 0: Generated reward right sound PSTH
Unit 0: Generated reward left sound PSTH
Unit 0: Generated nonreward right sound PSTH
Unit 0: Generated nonreward left sound PSTH
Processing unit 1...
Unit 1: Generated all right sound PSTH
Unit 1: Generated all left sound PSTH
Unit 1: Generated reward right sound PSTH
Unit 1: Generated reward left sound PSTH
Unit 1: Generated nonreward right sound PSTH
Unit 1: Generated nonreward left sound PSTH
Processing unit 2...
Unit 2: Skippi